# PQT vision-lab: detections on a Colab GPU

Runs `pqt-vision detect` over **consented research clips** in Google Drive and writes resumable Parquet
detection caches back to Drive. Everything after detection (tracking, zones, events, scoring) then runs on
CPU in Codespaces from those caches with the replay backend.

Rules (docs/privacy/vision-data.md, ADR-013):
- Only clips covered by a signed consent form that names Google Drive and Colab. **Never MOT17** or other
  non-commercial datasets here.
- Nothing pushes to GitHub from this notebook, and no tokens are needed: the repository is public.
- Compute units burn while a GPU runtime is attached, even idle: when the run ends, use
  **Runtime → Disconnect and delete runtime**.

In [ ]:
# Parameters: fill these in for every run.
REPO = "https://github.com/annsolomon/pqtone.git"
COMMIT = ""        # required: the full 40-character commit SHA to run (pin it; never a branch)
RUN_ID = ""        # e.g. "20261020-yolox_s"; re-use the same id to resume an interrupted run
CLIPS_DIR = "/content/drive/MyDrive/pqt/clips"
RUNS_DIR = "/content/drive/MyDrive/pqt/runs"
MODEL = "yolox_s"  # yolox_s ships (ADR-012); yolox_m / yolox_l are V7 challengers
CONF = 0.05        # cache at a low threshold; the threshold is chosen later, on CPU

In [ ]:
import re
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT), "set COMMIT to a full commit SHA"
assert re.fullmatch(r"[A-Za-z0-9._-]{1,64}", RUN_ID), "set RUN_ID"

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

In [ ]:
import os, subprocess
if not os.path.isdir("/content/pqtone"):
    subprocess.run(["git", "clone", "-q", REPO, "/content/pqtone"], check=True)
subprocess.run(["git", "-C", "/content/pqtone", "fetch", "-q", "origin", COMMIT], check=True)
subprocess.run(["git", "-C", "/content/pqtone", "checkout", "-q", COMMIT], check=True)
subprocess.run(["pip", "install", "-q", "-e", "/content/pqtone/pqt/vision/lab"], check=True)
# The GPU build of ONNX Runtime (MIT) replaces the CPU one; same version as pyproject.toml.
subprocess.run(["pip", "uninstall", "-q", "-y", "onnxruntime"], check=True)
subprocess.run(["pip", "install", "-q", "onnxruntime-gpu==1.20.1"], check=True)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
subprocess.run(["pqt-vision", "fetch-model", MODEL], check=True)

In [ ]:
import json, pathlib, time
clips = sorted(p for p in pathlib.Path(CLIPS_DIR).iterdir() if p.suffix.lower() in {".mp4", ".mkv", ".avi", ".mov"})
assert clips, f"no clips in {CLIPS_DIR}"
run = pathlib.Path(RUNS_DIR) / RUN_ID
run.mkdir(parents=True, exist_ok=True)
timings = {}
for clip in clips:
    t0 = time.time()
    subprocess.run(["pqt-vision", "detect", "--source", str(clip), "--model", MODEL, "--out", str(run / clip.stem),
                    "--conf", str(CONF), "--device", "cuda"], check=True)  # resumes if the cache exists
    timings[clip.name] = round(time.time() - t0, 1)
    print(clip.name, timings[clip.name], "s")

In [ ]:
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
(run / "run.json").write_text(json.dumps({"commit": COMMIT, "runId": RUN_ID, "model": MODEL, "conf": CONF,
                                          "gpu": gpu, "clips": [c.name for c in clips], "seconds": timings}, indent=2))
print("Done:", run)

## Next

1. Download `MyDrive/pqt/runs/<RUN_ID>/` and place it in `pqt/vision/lab/data/runs/<RUN_ID>/` in Codespaces
   (git-ignored).
2. Check it: `pqt-vision cache-info vision/lab/data/runs/<RUN_ID>/<clip>`.
3. **Runtime → Disconnect and delete runtime.**